# Resource Allocation for Data Science Labs

**SOTA Techniques:** Queueing Theory (M/M/c), Multi-armed Bandits, Bayesian Cost-Benefit

---

## Overview

Advanced resource allocation for managed data science environments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Load Project Data

In [ ]:
data_dir = '../data/synthetic'
try:
    df = pd.read_csv(f'{data_dir}/projects.csv')
    print(f'Loaded {len(df)} projects')
except:
    np.random.seed(42)
    types = ['data-analysis', 'ml-modeling', 'deep-learning', 'nlp', 'visualization']
    n = 200
    df = pd.DataFrame({
        'project_id': [f'PRJ{i:03d}' for i in range(n)],
        'project_type': np.random.choice(types, n),
        'team_size': np.random.randint(1, 20, n),
        'gpu_hours': np.random.exponential(100, n),
        'cpu_hours': np.random.exponential(500, n),
        'storage_gb': np.random.exponential(100, n),
        'duration_days': np.random.randint(1, 90, n),
        'status': np.random.choice(['active', 'completed', 'pending'], n)
    })
    print(f'Created {len(df)} synthetic projects')
print(df.head())

## 2. Resource Usage Analysis

In [ ]:
resource_summary = df.groupby('project_type').agg({
    'gpu_hours': 'mean',
    'cpu_hours': 'mean',
    'storage_gb': 'mean',
    'team_size': 'mean'
}).reset_index()
print('Average resource usage by project type:')
print(resource_summary)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].bar(resource_summary['project_type'], resource_summary['gpu_hours'])
axes[0,0].set_ylabel('Avg GPU Hours')
axes[0,0].set_title('GPU Usage by Type')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,1].bar(resource_summary['project_type'], resource_summary['cpu_hours'])
axes[0,1].set_ylabel('Avg CPU Hours')
axes[0,1].set_title('CPU Usage by Type')
axes[1,0].bar(resource_summary['project_type'], resource_summary['storage_gb'])
axes[1,0].set_ylabel('Avg Storage GB')
axes[1,0].set_title('Storage by Type')
axes[1,1].bar(resource_summary['project_type'], resource_summary['team_size'])
axes[1,1].set_ylabel('Avg Team Size')
axes[1,1].set_title('Team Size by Type')
plt.tight_layout()
plt.show()

## 3. Queueing Theory Model (M/M/c)

Model resource requests as a queueing system.

In [ ]:
arrivals_per_day = 1 / df['duration_days'].mean() * df['team_size'].mean()
service_rate = 1 / 7
print(f'Estimated arrival rate: {arrivals_per_day:.2f} projects/day')
print(f'Estimated service rate: {service_rate:.2f} projects/day')
lambda_r, mu = arrivals_per_day, service_rate
results = []
for servers in range(1, 8):
    rho = lambda_r / (servers * mu)
    if rho < 1:
        p0 = 1 / (sum((lambda_r/mu)**n / np.math.factorial(n) for n in range(servers)) +
                 (lambda_r/mu)**servers / np.math.factorial(servers) * 1/(1-rho))
        lq = (p0 * (lambda_r/mu)**servers * rho) / (np.math.factorial(servers) * (1-rho)**2)
        wq = lq / lambda_r
        results.append({'servers': servers, 'utilization': rho, 'avg_wait': wq})
metrics = pd.DataFrame(results)
print('Queueing metrics:')
print(metrics[['servers', 'utilization', 'avg_wait']])
plt.figure(figsize=(10, 4))
plt.plot(metrics['servers'], metrics['utilization'], 'o-', label='Utilization')
plt.plot(metrics['servers'], metrics['avg_wait'], 's-', label='Avg Wait Time')
plt.xlabel('Number of Resource Servers')
plt.ylabel('Metric Value')
plt.title('M/M/c Queueing Model Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Project Clustering for Resource Groups

In [ ]:
X = df[['gpu_hours', 'cpu_hours', 'storage_gb', 'team_size']].values
X_scaled = StandardScaler().fit_transform(X)
kmeans = KMeans(n_clusters=4, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)
print('Cluster centers:')
print(pd.DataFrame(kmeans.cluster_centers_, columns=['gpu', 'cpu', 'storage', 'team']))
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='gpu_hours', y='cpu_hours', hue='cluster', palette='viridis', alpha=0.6, s=50)
plt.xlabel('GPU Hours')
plt.ylabel('CPU Hours')
plt.title('Project Clusters')
plt.grid(True, alpha=0.3)
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

## 5. Resource Optimization with Bandits

In [ ]:
class ResourceBandit:
    def __init__(self, num_resources):
        self.num_resources = num_resources
        self.rewards = np.zeros(num_resources)
        self.counts = np.ones(num_resources)
    def select(self):
        theta = self.rewards / self.counts
        noise = np.random.normal(0, 0.1, self.num_resources)
        return np.argmax(theta + noise)
    def update(self, resource, reward):
        self.rewards[resource] += reward
        self.counts[resource] += 1

bandit = ResourceBandit(num_resources=4)
resource_names = ['GPU', 'CPU', 'Storage', 'Memory']
allocations = []
for _ in range(100):
    resource = bandit.select()
    reward = np.random.uniform(0.5, 1.5)
    bandit.update(resource, reward)
    allocations.append((resource_names[resource], reward))
alloc_counts = pd.Series([a[0] for a in allocations]).value_counts()
print('Resource allocations:')
print(alloc_counts)
plt.figure(figsize=(6, 4))
alloc_counts.plot(kind='bar')
plt.xlabel('Resource Type')
plt.ylabel('Number of Allocations')
plt.title('Resource Allocation Distribution')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Cost-Benefit Analysis

In [ ]:
df['cost_efficiency'] = df['duration_days'] / (df['gpu_hours'] + df['cpu_hours']/10 + df['storage_gb']/100)
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='project_type', y='cost_efficiency')
plt.xlabel('Project Type')
plt.ylabel('Cost Efficiency')
plt.title('Resource Cost Efficiency by Project Type')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()
print('Best performing project types:')
print(df.groupby('project_type')['cost_efficiency'].median().sort_values(ascending=False).head(3))

## Summary

This notebook demonstrated:
1. **Queueing theory models** (M/M/c)
2. **Project clustering** for resource grouping
3. **Multi-armed bandits** for allocation
4. **Cost-benefit analysis**